In [ ]:
# Step 2: Import required libraries
import json
import bz2
from elasticsearch import Elasticsearch, helpers
import pandas as pd

# Step 3: Connect to Elasticsearch
es = Elasticsearch(['http://elastic:9200'])

# Step 4: Create the Index with Zero Replicas
def create_index():
    es.options(ignore_status=[400]).indices.create(
        index='wikidata',
        body={
            'settings': {
                'number_of_shards': 1,
                'number_of_replicas': 0
            }
        }
    )

# Step 5: Function to parse the Wikidata JSON dump and extract labels
def parse_wikidata_dump(file_path, limit):
    count = 0
    with bz2.open(file_path, 'rt', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line.endswith(','):
                line = line[:-1]
            if line in ['[', ']']:
                continue

            try:
                item = json.loads(line)
                if 'labels' in item:
                    labels = item['labels']
                    if 'en' in labels:  # Extract English labels
                        yield {
                            '_index': 'wikidata',
                            '_source': {
                                'id': item['id'],
                                'label': labels['en']['value']
                            }
                        }
                        count += 1
                        if count >= limit:
                            break
            except json.JSONDecodeError as e:
                print(f"Error decoding JSON: {e}")

# Step 6: Index data into Elasticsearch
def index_wikidata(file_path, limit):
    actions = parse_wikidata_dump(file_path, limit)
    helpers.bulk(es, actions)

# Step 7: Create the index, load, and index the data
create_index()
wikidata_dump_file = 'my-data/latest-all.json.bz2'
sample_size = 1000  # Number of items to index
index_wikidata(wikidata_dump_file, sample_size)

# Step 8: Verify the indexed data
res = es.search(index="wikidata", body={"query": {"match_all": {}}, "size": sample_size})
for hit in res['hits']['hits']:
    print(hit['_source'])

In [ ]:
import pandas as pd
df = next(pd.read_csv("./my-data/organization_descriptions.csv", chunksize=100))

In [ ]:
df

In [ ]:
import pandas as pd
df = next(pd.read_csv("./my-data/organizations.csv", chunksize=100))
df

In [ ]:
import bz2
import json
import os
import sys
import traceback
from pymongo import MongoClient
from tqdm import tqdm
from datetime import datetime


def create_indexes(db):
    # Specify the collections and their respective fields to be indexed
    index_specs = {
        'cache': ['cell', 'lastAccessed', "limit"],  # Example: Indexing 'cell' and 'type' fields in 'cache' collection
        'items': ['id_entity', 'entity', 'category', 'popularity'],
        'literals': ['id_entity', 'entity'],
        'types': ['id_entity', 'entity']
    }

    for collection, fields in index_specs.items():
        if collection == "cache":
            db[collection].create_index(
                [
                    ("name", 1),
                    ("limit", 1),
                    ("kg", 1),
                    ("fuzzy", 1),
                    ("types", 1),
                    ("kind", 1),
                    ("NERtype", 1),
                    ("language", 1),
                ],
                unique=True,
                background=True,  # Create the index in the background
            )
        elif collection == "items":
            db[collection].create_index([('entity', 1), ('kind', 1)], unique=True)    
        for field in fields:
            db[collection].create_index([(field, 1)])  # 1 for ascending order


# MongoDB connection setup
MONGO_ENDPOINT, MONGO_ENDPOINT_PORT = os.environ["MONGO_ENDPOINT"].split(":")
MONGO_ENDPOINT_PORT = int(MONGO_ENDPOINT_PORT)
current_date = datetime.now()
formatted_date = current_date.strftime("%d%m%Y")
DB_NAME = f"crunchbase"

client = MongoClient(MONGO_ENDPOINT, MONGO_ENDPOINT_PORT)
log_c = client[DB_NAME].log
items_c = client[DB_NAME].items
literals_c = client[DB_NAME].literals
types_c = client[DB_NAME].types

c_ref = {
    "items": items_c,
    "literals":literals_c, 
    "types":types_c
}

create_indexes(client[DB_NAME])

buffer = {
    "items": [],
    "literals": [], 
    "types": []
}


def flush_buffer(buffer):
    for key in buffer:
        if len(buffer[key]) > 0:
            c_ref[key].insert_many(buffer[key])
            buffer[key] = []
            

def classify_value(value):
    try:
        # Check if value is a datetime
        dateutil.parser.isoparse(value)
        return 'DATETIME'
    except (ValueError, TypeError):
        pass
    try:
        # Check if value is a number
        float(value)
        return 'NUMBER'
    except (ValueError, TypeError):
        pass
    # If neither, it's a string
    return 'STRING'
    
def parse_data(index, columns, data, addional_data):
    objects = {}
    literals = {datatype: {} for datatype in ["STRING", "DATETIME", "NUMBER"]}
    types = {"P31": ["Organization"]}
    join = {
        "items": {
            "id_entity": i,
            "entity": entity,
            "description": description,
            "labels": all_labels,
            "aliases": all_aliases,
            "types": types,
            "popularity": popularity,
            "kind": "entity",
            "NERtype": "ORG"
        },
        "objects": { 
            "id_entity": i,
            "entity": entity,
            "objects":objects
        },
        "literals": { 
            "id_entity": i,
            "entity": entity,
            "literals": literals
        },
        "types": { 
            "id_entity": i,
            "entity": entity,
            "types": types
        },
    }

    

    for key in buffer:
        buffer[key].append(join[key])            

    if len(buffer["items"]) == BATCH_SIZE:
        flush_buffer(buffer)


           
# Read large CSV file in chunks
chunk_size = 1000  # Adjust chunk size as needed
file_path = './my-data/organizations.csv'  # Update with your file path

# Determine the number of chunks for progress bar
total_lines = sum(1 for _ in open(file_path))
total_chunks = total_lines // chunk_size + (1 if total_lines % chunk_size != 0 else 0)
index = 0
# Process the file in chunks
with tqdm(total=total_chunks, desc="Processing") as pbar:
    for chunk in pd.read_csv(file_path, chunksize=chunk_size):
        #process_and_insert(chunk, items)
        columns = chunk.columns
        for _, data in chunk.iterrows():
            for column in columns:
                print(data[column])  
        pbar.update(1)

print("Finished processing and inserting documents.")

In [ ]:
columns = chunk.columns
for _, data in chunk.iterrows():
    for column in columns:
        print(data[column])   
    break

In [ ]:
import pandas as pd
df = next(pd.read_csv("../data/organizations.csv", chunksize=100))
df

In [ ]:
import pandas as pd
df = next(pd.read_csv("./my-data/organization_descriptions.csv", chunksize=100))
df

In [ ]:
# Read large CSV file in chunks
chunk_size = 1000  # Adjust chunk size as needed
file_path = './my-data/organization_descriptions.csv'  # Update with your file path

# Determine the number of chunks for progress bar
total_lines = sum(1 for _ in open(file_path))
total_chunks = total_lines // chunk_size + (1 if total_lines % chunk_size != 0 else 0)
index = 0
data = {}
# Process the file in chunks
with tqdm(total=total_chunks, desc="Processing") as pbar:
    for chunk in pd.read_csv(file_path, chunksize=chunk_size):
        columns = chunk.columns
        for _, data in chunk.iterrows():
            id = data["uuid"]
            url = data["cb_url"]
            popularity = data["rank"]
            description = data["description"]
            data[id] = {
                "url": url,  
                "description": description,
                "popularity": popularity
            }
        pbar.update(1)

In [ ]:
import pandas as pd
df = pd.read_csv("../data/organizations.csv")

In [ ]:
df["rank"]

In [ ]:
int(df["rank"].mean())

In [ ]:
! pip install column-classifier==0.1.0

In [ ]:
! python -m spacy download en_core_web_trf

In [ ]:
from column_classifier.column_classifier import ColumnClassifier

classifier = ColumnClassifier()